# INR/USD Fair Value Model — Data & Estimation

This notebook builds the complete dataset and estimates three equilibrium models for INR/USD:

1. **REER Structural Anchor** — long-run real exchange rate equilibrium via HP-filtered REER trend
2. **FEER Sustainability Engine** — fundamental equilibrium with a novel conditional financing norm
3. **BEER + Regime Engine** — short-run behavioural equilibrium with Markov-switching regime classification

The dataset spans January 2000 to April 2026, using data from FRED, the RBI's DBIE portal, and the World Bank.
The pipeline applies point-in-time publication lags to every input so that backtests reflect only what was actually knowable at each date.


## 1. Data Refresh: FRED Series + GDP Extrapolation

Pull the latest US/global market data from FRED and extrapolate India's nominal GDP forward using
the RBI's 6.9% real growth projection for FY2026-27, converted to USD at each month's actual INR/USD
rate to preserve the FX channel.


In [ ]:
import pandas as pd
import numpy as np
from fredapi import Fred
import shutil

RAW       = '../data/raw/'
PROCESSED = '../data/processed/'

FRED_API_KEY = 'YOUR_FRED_API_KEY'     # get one at https://fred.stlouisfed.org/docs/api/api_key.html
fred = Fred(api_key=FRED_API_KEY)
START, END = '2000-01-01', '2026-06-01'

# GDP extrapolation anchored to RBI's own growth projection
RBI_REAL_GROWTH = 0.069
GDP_DEFLATOR    = 0.040
NOMINAL_GROWTH  = (1 + RBI_REAL_GROWTH) * (1 + GDP_DEFLATOR) - 1
MONTHLY_FACTOR  = (1 + NOMINAL_GROWTH) ** (1/12)

def to_ms(s):
    s = s.resample('MS').last()
    s.index = s.index.to_period('M').to_timestamp()
    return s

# back up the existing master before touching it
shutil.copy(f'{PROCESSED}master.csv', f'{PROCESSED}master_backup_prelag.csv')

master = pd.read_csv(f'{PROCESSED}master.csv', index_col=0, parse_dates=True)
master.index = pd.to_datetime(master.index).to_period('M').to_timestamp()
master = master[~master.index.duplicated(keep='last')]
master_idx = master.index

# --- refresh market/global series (not cpi_india — that has its own splice) ---
fred_map = {
    'CPIAUCSL'        : 'cpi_us',
    'FEDFUNDS'        : 'fed_funds_rate',
    'DTWEXBGS'        : 'dxy',            # broad trade-weighted, NOT ICE DXY
    'VIXCLS'          : 'vix',
    'GS10'            : 'us_10y_yield',
    'DCOILBRENTEU'    : 'brent',
    'WALCL'           : 'fed_balance_sheet',
    'IRSTCI01INM156N' : 'india_stir',
}
for sid, col in fred_map.items():
    try:
        s = to_ms(fred.get_series(sid, observation_start=START, observation_end=END))
        master[col] = s.reindex(master_idx)
        print(f"  {col:<22} ends {s.dropna().index[-1].strftime('%b %Y')}")
    except Exception as e:
        print(f"  {col:<22} FAILED — {e}")

# backfill DXY with the older narrow index for 2000-2006
try:
    dxy_n = to_ms(fred.get_series('DTWEXM', observation_start=START, observation_end='2006-01-01'))
    master['dxy'] = master['dxy'].combine_first(dxy_n.reindex(master_idx))
except Exception:
    pass

# patch the dead repo-rate series with india_stir
gap = master['rbi_repo_rate'].isnull()
master.loc[gap, 'rbi_repo_rate'] = master.loc[gap, 'india_stir']


### GDP Extrapolation

The World Bank's GDP series ends at 2024. We grow nominal INR GDP at 11.18%/year
(RBI real 6.9% × deflator 4%), then convert to USD at each month's spot rate.
This means USD GDP barely grows during depreciation episodes — the FX channel works correctly.


In [ ]:
wb_gdp = pd.read_csv(f'{RAW}wb_gdp_india.csv', index_col=0, parse_dates=True).squeeze()
wb_gdp.index = pd.to_datetime(wb_gdp.index).to_period('M').to_timestamp()
wb_gdp = wb_gdp[~wb_gdp.index.duplicated(keep='last')].dropna()

anchor_year    = wb_gdp.index.max().year
anchor_usd_gdp = float(wb_gdp.iloc[-1])
avg_rate       = master.loc[master.index.year == anchor_year, 'inr_usd'].dropna().mean()
anchor_inr_gdp = anchor_usd_gdp * avg_rate
anchor_ref     = pd.Timestamp(year=anchor_year, month=12, day=1)

print(f"Anchor: {anchor_year} GDP = ${anchor_usd_gdp/1e12:.3f}T, avg INR {avg_rate:.2f}")
print(f"Growing at {NOMINAL_GROWTH*100:.2f}% nominal, FX-adjusted monthly\n")

for dt in master_idx:
    if dt <= anchor_ref:
        continue
    rate = master.loc[dt, 'inr_usd']
    if pd.isnull(rate):
        continue
    months = (dt.year - anchor_ref.year) * 12 + (dt.month - anchor_ref.month)
    master.loc[dt, 'gdp_india_usd'] = anchor_inr_gdp * (MONTHLY_FACTOR ** months) / rate

# recompute derived series that depend on the refresh
master['cpi_us_yoy']    = master['cpi_us'].pct_change(12, fill_method=None) * 100
master['real_rate_us']  = master['fed_funds_rate'] - master['cpi_us_yoy']
master['real_rate_diff'] = master['real_rate_india'] - master['real_rate_us']
master['brent_yoy']     = master['brent'].pct_change(12, fill_method=None) * 100
master['fed_bs_yoy']    = master['fed_balance_sheet'].pct_change(12, fill_method=None) * 100
master['cab_pct_gdp']   = ((master['current_account'] * 4) / (master['gdp_india_usd'] / 1e6)) * 100
master['remit_pct_gdp'] = (master['remittances_usd'] / master['gdp_india_usd']) * 100

master.to_csv(f'{PROCESSED}master.csv')
print(f"master.csv saved  ({master.shape[0]} × {master.shape[1]})")


### Brent: Switch to Month-Average

FRED's daily Brent series can produce misleading month-end prints during volatile periods.
We switch to monthly *averages*, matching the IMF convention used to validate the oil elasticity.


In [ ]:
# inspect daily data to confirm month-average is more stable
daily = fred.get_series('DCOILBRENTEU', observation_start='2026-01-01', observation_end='2026-06-01')
md_grp = daily.resample('MS')
print(pd.DataFrame({'mean': md_grp.mean(), 'last': md_grp.last(), 'max': md_grp.max()}).round(2))


In [ ]:
# re-pull full history as monthly average
brent_daily = fred.get_series('DCOILBRENTEU', observation_start='2000-01-01', observation_end='2026-06-01')
brent_avg = brent_daily.resample('MS').mean()
brent_avg.index = brent_avg.index.to_period('M').to_timestamp()

master = pd.read_csv(f'{PROCESSED}master.csv', index_col=0, parse_dates=True)
master.index = pd.to_datetime(master.index).to_period('M').to_timestamp()
master['brent'] = brent_avg.reindex(master.index)
master['brent_yoy'] = master['brent'].pct_change(12, fill_method=None) * 100
master.to_csv(f'{PROCESSED}master.csv')

print("Last 6 months (month-average):")
print(master['brent'].dropna().tail(6).round(1).to_string())


### Lock Brent Normal

The structural Brent anchor for the FEER oil adjustment. We use the full modeling-window median (2004-2025),
fixed rather than rolling, to exclude the pre-2004 cheap-oil era and transient geopolitical spikes.


In [ ]:
b = master['brent'].dropna()
print(f"Modeling window median (2004-Jun25) : {b['2004-04':'2025-06'].median():.1f}")
print(f"Prior value (Phases 2-4)            : 72.3")
print(f"Full-sample median (2000-2026)      : {b.median():.1f}")

BRENT_NORM = 72.0   # locked


## 2. RBI Data Refresh + BoP Financing Panel

Parse the six RBI DBIE files, extract the full Balance of Payments quarterly panel
(9 components: current account, merchandise, private transfers, capital account, FDI, portfolio,
ECB/loans, banking capital, reserve movements), and splice FPI flows from the BoP definition
starting March 2011.


In [ ]:
import warnings; warnings.filterwarnings('ignore')
import openpyxl
from datetime import datetime

MM = {'JAN':1,'FEB':2,'MAR':3,'APR':4,'MAY':5,'JUN':6,'JUL':7,'AUG':8,
      'SEP':9,'OCT':10,'NOV':11,'DEC':12,
      'JANUARY':1,'FEBRUARY':2,'MARCH':3,'APRIL':4,'JUNE':6,'JULY':7,
      'AUGUST':8,'SEPTEMBER':9,'OCTOBER':10,'NOVEMBER':11,'DECEMBER':12}
QM = {'Q1':4,'Q2':7,'Q3':10,'Q4':1}

def ms_index(d):
    s = pd.Series(d) if isinstance(d, dict) else d
    s.index = pd.to_datetime(s.index).to_period('M').to_timestamp()
    return s.sort_index()

# --- INR/USD ---
wb = openpyxl.load_workbook(RAW+'rbi_inr_usd.xlsx', data_only=True); ws=wb.active
d = {}
for r in ws.iter_rows(min_row=9, values_only=True):
    dt, v = r[1], r[5]
    if isinstance(dt, datetime) and isinstance(v, (int,float)): d[dt] = float(v)
inr = ms_index(d)

# --- REER (40-currency, 2015-16=100) ---
wb = openpyxl.load_workbook(RAW+'rbi_reer.xlsx', data_only=True); ws=wb.active
d = {}
for r in ws.iter_rows(min_row=8, values_only=True):
    dt, v = r[1], r[3]
    if isinstance(dt, datetime) and isinstance(v, (int,float)): d[dt] = float(v)
reer = ms_index(d)

# --- Reserves (year carry-forward for merged cells) ---
wb = openpyxl.load_workbook(RAW+'rbi_reserves.xlsx', data_only=True); ws=wb.active
d = {}; cy = None
for r in ws.iter_rows(min_row=7, values_only=True):
    y = r[1]
    if y is not None:
        ys = str(y).strip()
        if ys.replace('.0','').isdigit(): cy = int(float(ys))
    m, tot = r[2], r[13]
    if cy and isinstance(m,str) and m.strip().upper() in MM and isinstance(tot,(int,float)):
        d[pd.Timestamp(year=cy, month=MM[m.strip().upper()], day=1)] = float(tot)
res = ms_index(d)

# --- Trade ---
wb = openpyxl.load_workbook(RAW+'rbi_trade.xlsx', data_only=True); ws=wb.active
rows = []
for r in ws.iter_rows(min_row=8, values_only=True):
    fy, m, ex, im = r[1], r[2], r[3], r[4]
    if (isinstance(fy,str) and '-' in fy and isinstance(m,str) and m.strip().upper() in MM
        and isinstance(ex,(int,float)) and isinstance(im,(int,float))):
        mo = MM[m.strip().upper()]
        ys = int(fy.split('-')[0]); yr = ys if mo >= 4 else ys + 1
        rows.append((pd.Timestamp(year=yr, month=mo, day=1), float(ex), float(im)))
tr = pd.DataFrame(rows, columns=['d','exports_usd_mn','imports_usd_mn']).set_index('d')
tr = tr[~tr.index.duplicated(keep='last')].sort_index()
tr['trade_balance_usd_mn'] = tr['exports_usd_mn'] - tr['imports_usd_mn']

# --- CAB full BoP panel (Net rows only) ---
wb = openpyxl.load_workbook(RAW+'rbi_cab.xlsx', data_only=True); ws=wb.active
COL = {'ca':5, 'merch':6, 'priv_transfer':20, 'capacct':24,
       'fdi_bop':26, 'portfolio_bop':35, 'loans':42, 'banking':52, 'reserve_mov':63}
rec = {}
for r in ws.iter_rows(min_row=8, values_only=True):
    fy, q, tx = r[1], r[2], r[3]
    if (isinstance(fy,str) and '-' in fy and isinstance(q,str) and q.strip() in QM
        and isinstance(tx,str) and tx.strip().lower() == 'net'):
        ys = int(fy.split('-')[0]); mo = QM[q.strip()]
        yr = ys if mo >= 4 else ys + 1
        rec[pd.Timestamp(year=yr, month=mo, day=1)] = {
            k: (float(r[i]) if isinstance(r[i],(int,float)) else np.nan) for k, i in COL.items()}
cab = pd.DataFrame(rec).T.sort_index()

# --- Monthly FPI/FDI from BoP foreign investment table ---
wb = openpyxl.load_workbook(RAW+'rbi_fpi_flows.xlsx', data_only=True); ws=wb.active
d_pf = {}; d_fdi = {}
for r in ws.iter_rows(min_row=7, values_only=True):
    lbl = r[1]
    if isinstance(lbl,str) and ':' in lbl and '(' in lbl:
        try:
            yr = int(lbl.split(':')[0]); mo = int(lbl.split(':')[1].split('(')[0])
            dt = pd.Timestamp(year=yr, month=mo, day=1)
            if isinstance(r[20],(int,float)): d_pf[dt]  = float(r[20])
            if isinstance(r[2], (int,float)): d_fdi[dt] = float(r[2])
        except: pass
new_portfolio = ms_index(d_pf)
new_fdi       = ms_index(d_fdi)

# --- Splice FPI at March 2011: pre-2011 = old FII definition, post = BoP Net Portfolio ---
old = pd.read_csv(PROCESSED+'master.csv', index_col=0, parse_dates=True)
old.index = pd.to_datetime(old.index).to_period('M').to_timestamp()
seam = pd.Timestamp('2011-03-01')
fpi_usd = pd.concat([old['fpi_flows_usd_mn'].dropna().loc[:seam - pd.DateOffset(months=1)],
                      new_portfolio.loc[seam:]]).sort_index()
fpi_usd = fpi_usd[~fpi_usd.index.duplicated(keep='last')]
fpi_inr = (fpi_usd * inr.reindex(fpi_usd.index) / 10.0).dropna()

# --- Assemble raw series on monthly grid ---
idx = pd.date_range('2000-01-01', '2026-05-01', freq='MS')
out = pd.DataFrame(index=idx); out.index.name = 'date'
out['inr_usd']              = inr.reindex(idx)
out['reer']                 = reer.reindex(idx)
out['reserves_usd_mn']      = res.reindex(idx)
out['exports_usd_mn']       = tr['exports_usd_mn'].reindex(idx)
out['imports_usd_mn']       = tr['imports_usd_mn'].reindex(idx)
out['trade_balance_usd_mn'] = tr['trade_balance_usd_mn'].reindex(idx)
for c in COL: out[c]        = cab[c].reindex(idx)
out['fpi_flows_usd_mn']     = fpi_usd.reindex(idx)
out['fpi_flows_inr_cr']     = fpi_inr.reindex(idx)
out['fdi_net_usd_mn']       = new_fdi.reindex(idx)
out.to_csv(PROCESSED+'step3a_raw.csv')

# quick inspection
print("Series edges:")
for c in out.columns:
    s = out[c].dropna()
    if len(s): print(f"  {c:<24} {s.index[0].strftime('%b%Y')} → {s.index[-1].strftime('%b%Y')}  (n={len(s)})")


### Merge into Master Dataset

Overwrite the RBI-sourced columns in master with the freshly parsed values, add the 9 new BoP panel
columns and the monthly FDI series, then recompute all derived ratios.


In [ ]:
import shutil

master = pd.read_csv(PROCESSED+'master.csv', index_col=0, parse_dates=True)
master.index = pd.to_datetime(master.index).to_period('M').to_timestamp()
master = master[~master.index.duplicated(keep='last')]

raw = pd.read_csv(PROCESSED+'step3a_raw.csv', index_col=0, parse_dates=True)
raw.index = pd.to_datetime(raw.index).to_period('M').to_timestamp()

idx = pd.date_range(min(master.index.min(), raw.index.min()),
                    max(master.index.max(), raw.index.max()), freq='MS')
master, raw = master.reindex(idx), raw.reindex(idx)

# pull NEER (trade-weighted, col 2)
wb = openpyxl.load_workbook(RAW+'rbi_reer.xlsx', data_only=True); ws=wb.active
d_neer = {}
for r in ws.iter_rows(min_row=8, values_only=True):
    dt, v = r[1], r[2]
    if isinstance(dt, datetime) and isinstance(v, (int,float)): d_neer[dt] = float(v)
neer = pd.Series(d_neer); neer.index = pd.to_datetime(neer.index).to_period('M').to_timestamp()
master['neer'] = neer.sort_index().reindex(idx)

# overwrite RBI columns
for mc, rc in {'inr_usd':'inr_usd', 'reer':'reer', 'fx_reserves_usd_mn':'reserves_usd_mn',
               'current_account':'ca', 'exports_usd_mn':'exports_usd_mn',
               'imports_usd_mn':'imports_usd_mn', 'trade_balance':'trade_balance_usd_mn',
               'fpi_flows_usd_mn':'fpi_flows_usd_mn', 'fpi_flows_inr_cr':'fpi_flows_inr_cr'}.items():
    master[mc] = raw[rc]

# add new BoP panel columns
for mc, rc in {'merch_balance_usd_mn':'merch', 'priv_transfer_usd_mn':'priv_transfer',
               'capital_account_usd_mn':'capacct', 'fdi_bop_usd_mn':'fdi_bop',
               'portfolio_bop_usd_mn':'portfolio_bop', 'loans_usd_mn':'loans',
               'banking_usd_mn':'banking', 'reserve_mov_usd_mn':'reserve_mov',
               'fdi_net_usd_mn':'fdi_net_usd_mn'}.items():
    master[mc] = raw[rc]

# recompute derived series
master['cpi_us_yoy']      = master['cpi_us'].pct_change(12, fill_method=None) * 100
master['inflation_diff']  = master['cpi_india_yoy'] - master['cpi_us_yoy']
master['real_rate_india'] = master['rbi_repo_rate'] - master['cpi_india_yoy']
master['real_rate_us']    = master['fed_funds_rate'] - master['cpi_us_yoy']
master['real_rate_diff']  = master['real_rate_india'] - master['real_rate_us']
master['justified_depreciation'] = master['inflation_diff']
master['brent_yoy']       = master['brent'].pct_change(12, fill_method=None) * 100
master['fed_bs_yoy']      = master['fed_balance_sheet'].pct_change(12, fill_method=None) * 100
master['inr_yoy_chg']     = master['inr_usd'].pct_change(12, fill_method=None) * 100
master['reserves_mom_chg']= master['fx_reserves_usd_mn'].diff()
master['import_cover']    = master['fx_reserves_usd_mn'] / master['imports_usd_mn']
master['cab_pct_gdp']     = (master['current_account']*4) / (master['gdp_india_usd']/1e6) * 100
master['remit_pct_gdp']   = master['remittances_usd'] / master['gdp_india_usd'] * 100
master['fpi_pct_gdp']     = (master['fpi_flows_usd_mn']*12) / (master['gdp_india_usd']/1e6) * 100

master = master.loc[:master.dropna(how='all').index.max()]
shutil.copy(PROCESSED+'master.csv', PROCESSED+'master_v1_post_step1.csv')
master.to_csv(PROCESSED+'master_v2.csv')
print(f"master_v2.csv: {master.shape[0]} × {master.shape[1]}, {master.index[0].strftime('%b %Y')} → {master.index[-1].strftime('%b %Y')}")


## 3. Point-in-Time Publication Lags

Apply each series' real-world publication delay, then recompute all derived ratios from the lagged inputs.
This is the single most important step for backtest integrity.


In [ ]:
m = pd.read_csv(PROCESSED+'master_v2.csv', index_col=0, parse_dates=True)
m.index = pd.to_datetime(m.index).to_period('M').to_timestamp()

LAG = {
    'cpi_us': 1, 'cpi_india': 1,
    'exports_usd_mn': 1, 'imports_usd_mn': 1, 'trade_balance': 1,
    'fpi_flows_usd_mn': 2, 'fpi_flows_inr_cr': 2, 'fdi_net_usd_mn': 2,
    'current_account': 6, 'merch_balance_usd_mn': 6, 'priv_transfer_usd_mn': 6,
    'capital_account_usd_mn': 6, 'fdi_bop_usd_mn': 6, 'portfolio_bop_usd_mn': 6,
    'loans_usd_mn': 6, 'banking_usd_mn': 6, 'reserve_mov_usd_mn': 6,
    'gdp_india_usd': 3, 'gdp_us_usd': 3, 'gdp_pc_india': 3, 'gdp_pc_us': 3,
    'remittances_usd': 3,
}

# extend index so shifted values aren't lost at the tail
max_lag = max(LAG.values())
new_idx = pd.date_range(m.index[0], m.index[-1] + pd.DateOffset(months=max_lag), freq='MS')
m = m.reindex(new_idx)

ml = m.copy()
for col, k in LAG.items():
    if col in ml.columns: ml[col] = m[col].shift(k)

# recompute derived from lagged inputs
ml['cpi_us_yoy']      = ml['cpi_us'].pct_change(12, fill_method=None) * 100
ml['cpi_india_yoy']   = ml['cpi_india'].pct_change(12, fill_method=None) * 100
ml['inflation_diff']  = ml['cpi_india_yoy'] - ml['cpi_us_yoy']
ml['real_rate_india'] = ml['rbi_repo_rate'] - ml['cpi_india_yoy']
ml['real_rate_us']    = ml['fed_funds_rate'] - ml['cpi_us_yoy']
ml['real_rate_diff']  = ml['real_rate_india'] - ml['real_rate_us']
ml['justified_depreciation'] = ml['inflation_diff']
ml['brent_yoy']       = ml['brent'].pct_change(12, fill_method=None) * 100
ml['fed_bs_yoy']      = ml['fed_balance_sheet'].pct_change(12, fill_method=None) * 100
ml['inr_yoy_chg']     = ml['inr_usd'].pct_change(12, fill_method=None) * 100
ml['reserves_mom_chg']= ml['fx_reserves_usd_mn'].diff()
ml['import_cover']    = ml['fx_reserves_usd_mn'] / ml['imports_usd_mn']
ml['cab_pct_gdp']     = (ml['current_account']*4) / (ml['gdp_india_usd']/1e6) * 100
ml['remit_pct_gdp']   = ml['remittances_usd'] / ml['gdp_india_usd'] * 100
ml['fpi_pct_gdp']     = (ml['fpi_flows_usd_mn']*12) / (ml['gdp_india_usd']/1e6) * 100

ml = ml.loc[:ml.dropna(how='all').index.max()]
ml.to_csv(PROCESSED+'master_v2_lagged.csv')
print(f"master_v2_lagged.csv: {ml.shape}, {ml.index[0].strftime('%b %Y')} → {ml.index[-1].strftime('%b %Y')}")


## 4. Model 1: REER Structural Anchor

HP-filter the 40-currency REER to extract the long-run trend, then compute the misalignment
(how far the spot rate deviates from where the trend REER implies it should be).
Positive misalignment = INR undervalued (weaker than fair).


In [ ]:
from statsmodels.tsa.filters.hp_filter import hpfilter

m = pd.read_csv(PROCESSED+'master_v2_lagged.csv', index_col=0, parse_dates=True)
m.index = pd.to_datetime(m.index).to_period('M').to_timestamp()

reer = m['reer'].dropna()
inr  = m['inr_usd'].reindex(reer.index).dropna()
common = reer.index.intersection(inr.index)
reer, inr = reer.loc[common], inr.loc[common]

cycle, trend = hpfilter(reer.values, lamb=129600)
reer_trend = pd.Series(trend, index=reer.index)
misalign   = (reer_trend / reer - 1) * 100
inr_fair   = inr * (reer / reer_trend)

out = pd.DataFrame({
    'inr_usd': inr, 'reer': reer, 'reer_trend': reer_trend,
    'reer_deviation_pct': (reer / reer_trend - 1) * 100,
    'misalignment_pct': misalign, 'inr_fair_reer': inr_fair,
}, index=common)
out.to_csv(PROCESSED+'phase2_equilibrium_v2.csv')

# episode validation
episodes = {'GFC (Oct 2008)':'2008-10-01', 'Taper (Aug 2013)':'2013-08-01',
            'COVID (Apr 2020)':'2020-04-01', 'Fed tight (Oct 2022)':'2022-10-01',
            'Pre-war (Dec 2025)':'2025-12-01', 'Current (Apr 2026)':'2026-04-01'}
print(f"{'Episode':<24}{'REER':>7}{'Trend':>7}{'Misalign':>9}{'Fair':>8}")
for lbl, dt in episodes.items():
    dt = pd.Timestamp(dt)
    if dt in out.index:
        r = out.loc[dt]
        print(f"{lbl:<24}{r.reer:>7.1f}{r.reer_trend:>7.1f}{r.misalignment_pct:>+8.1f}%{r.inr_fair_reer:>8.2f}")


## 5. Model 2: FEER Sustainability + Conditional Financing Norm

Two versions of the Fundamental Equilibrium Exchange Rate:

- **Static FEER:** what rate makes the CAD sustainable at a fixed −2.5% norm?
- **Conditional FEER:** what rate is sustainable given the *current financing environment*?
  The conditional norm equals −(FDI + ECB loans) × 4 / GDP — the deficit that stable capital inflows can finance.
  Remittances are excluded (they're already inside the CAD, not financing it).


In [ ]:
import statsmodels.api as sm

REER_SEMI, OIL_ELAST, BRENT_NORM, CAD_NORM_STATIC = -0.267, -2.45, 72.0, -2.5

m = pd.read_csv(PROCESSED+'master_v2.csv', index_col=0, parse_dates=True)
m.index = pd.to_datetime(m.index).to_period('M').to_timestamp()
p2 = pd.read_csv(PROCESSED+'phase2_equilibrium_v2.csv', index_col=0, parse_dates=True)
p2.index = pd.to_datetime(p2.index).to_period('M').to_timestamp()

# build quarterly contemporaneous dataset
recs = []
for dt in m['current_account'].dropna().index:
    qm = pd.date_range(dt, periods=3, freq='MS')
    brent_q, reer_q, inr_q = [m.loc[m.index.isin(qm), c].mean() for c in ['brent','reer','inr_usd']]
    ca, gdp, fdi, loans_v = [m.loc[dt, c] for c in ['current_account','gdp_india_usd','fdi_bop_usd_mn','loans_usd_mn']]
    if pd.notna(gdp) and gdp > 0 and pd.notna(ca) and pd.notna(brent_q):
        recs.append({'date':dt, 'cad_pct':(ca*4)/(gdp/1e6)*100, 'brent_q':brent_q,
                     'reer_q':reer_q, 'inr_q':inr_q, 'fdi':fdi, 'loans':loans_v, 'gdp':gdp})
q = pd.DataFrame(recs).set_index('date').sort_index()
q['log_brent'] = np.log(q['brent_q'])

# oil adjustment + static FEER
q['oil_drag']        = OIL_ELAST * (q['log_brent'] - np.log(BRENT_NORM))
q['cad_underlying']  = q['cad_pct'] - q['oil_drag']
q['cad_gap_static']  = q['cad_underlying'] - CAD_NORM_STATIC
q['feer_static_pct'] = -q['cad_gap_static'] / REER_SEMI

# conditional FEER
q['stable_fin']     = q['fdi'] + q['loans']
q['stable_pct_gdp'] = (q['stable_fin'] * 4) / (q['gdp'] / 1e6) * 100
q['cad_norm_cond']  = np.where(q['stable_fin']>0, -q['stable_pct_gdp'], 0)
q['cad_gap_cond']   = q['cad_underlying'] - q['cad_norm_cond']
q['feer_cond_pct']  = -q['cad_gap_cond'] / REER_SEMI

q['inr_fair_static'] = q['inr_q'] / (1 + q['feer_static_pct']/100)
q['inr_fair_cond']   = q['inr_q'] / (1 + q['feer_cond_pct']/100)
q['financing_premium'] = q['feer_static_pct'] - q['feer_cond_pct']
q.to_csv(PROCESSED+'phase3_feer_v2.csv')

last = q.iloc[-1]
print(f"Latest quarter ({q.index[-1].strftime('%b %Y')}): CAD {last.cad_pct:.1f}%, Brent ${last.brent_q:.0f}")
print(f"  Static  FEER: {last.feer_static_pct:+.1f}%  fair={last.inr_fair_static:.2f}")
print(f"  Cond.   FEER: {last.feer_cond_pct:+.1f}%  fair={last.inr_fair_cond:.2f}")
print(f"  Financing premium: {last.financing_premium:+.1f}pp")


## 6. Model 3: BEER + Regime Engine

Cointegrating regression of log(INR) on five macro drivers, followed by a 2×2 Oil×DXY regime matrix
and a two-state Markov-switching model to classify calm vs stress episodes.


In [ ]:
import json
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression

ml = pd.read_csv(PROCESSED+'master_v2_lagged.csv', index_col=0, parse_dates=True)
ml.index = pd.to_datetime(ml.index).to_period('M').to_timestamp()

d = ml[['inr_usd','dxy','real_rate_diff','fpi_pct_gdp','brent','vix']].dropna().copy()
d['log_inr'], d['log_dxy'], d['log_brent'] = np.log(d['inr_usd']), np.log(d['dxy']), np.log(d['brent'])

# BEER regression
X = sm.add_constant(d[['log_dxy','real_rate_diff','fpi_pct_gdp','log_brent','vix']])
beer = sm.OLS(d['log_inr'], X).fit(cov_type='HAC', cov_kwds={'maxlags':6})
d['inr_fair_beer'] = np.exp(beer.fittedvalues)
d['beer_misalign'] = (d['inr_usd'] / d['inr_fair_beer'] - 1) * 100

adf = adfuller(beer.resid, maxlag=12, autolag='AIC')
dxy_r2 = sm.OLS(d['log_inr'], sm.add_constant(d[['log_dxy']])).fit().rsquared

print(f"BEER: R²={beer.rsquared:.3f}, ADF p={adf[1]:.4f}, DXY-only R²={dxy_r2:.3f}")
for v in ['log_dxy','real_rate_diff','fpi_pct_gdp','log_brent','vix']:
    print(f"  {v:<16} {beer.params[v]:+.4f}  p={beer.pvalues[v]:.4f}")

# Oil×DXY regime matrix
d['inr_ret'] = d['log_inr'].diff() * 100
brent_med, dxy_med = d['brent'].median(), d['log_dxy'].median()
d['regime_4'] = (np.where(d['brent']>brent_med,'Hi','Lo') + '_' +
                 np.where(d['log_dxy']>dxy_med,'Hi','Lo'))

print(f"\nRegime matrix (mean depreciation %/mo):")
for reg in ['Lo_Lo','Lo_Hi','Hi_Lo','Hi_Hi']:
    sub = d.loc[d.regime_4==reg, 'inr_ret'].dropna()
    print(f"  {reg}  {sub.mean():+.2f}%  n={len(sub)}")

# Markov switching
ms = MarkovRegression(d['inr_ret'].dropna(), k_regimes=2, trend='c', switching_variance=True)
ms_fit = ms.fit(maxiter=500, em_iter=200)
stress = 1 if np.sqrt(ms_fit.params['sigma2[1]']) > np.sqrt(ms_fit.params['sigma2[0]']) else 0
d['p_stress'] = ms_fit.smoothed_marginal_probabilities[stress].reindex(d.index)

mu_s = ms_fit.params[f'const[{stress}]']; sig_s = np.sqrt(ms_fit.params[f'sigma2[{stress}]'])
print(f"\nStress regime: μ={mu_s:+.2f}%/mo, σ={sig_s:.2f}%")
print(f"Last 3 months: {', '.join(f'{dt.strftime("%b%y")}={d.loc[dt,"p_stress"]:.2f}' for dt in d.index[-3:])}")

# save
d[['inr_usd','inr_fair_beer','beer_misalign','inr_ret','regime_4','p_stress']].to_csv(PROCESSED+'phase4_beer_v2.csv')
coefs = {v: float(beer.params[v]) for v in beer.params.index}
coefs['r_squared'], coefs['adf_p'] = float(beer.rsquared), float(adf[1])
with open(PROCESSED+'beer_coefs_v2.json','w') as f: json.dump(coefs, f, indent=2)


## Four-Model Convergence


In [ ]:
p2 = pd.read_csv(PROCESSED+'phase2_equilibrium_v2.csv', index_col=0, parse_dates=True)
p2.index = pd.to_datetime(p2.index).to_period('M').to_timestamp()
p3 = pd.read_csv(PROCESSED+'phase3_feer_v2.csv', index_col=0, parse_dates=True)
p3.index = pd.to_datetime(p3.index).to_period('M').to_timestamp()

r2, r3, r4 = p2.iloc[-1], p3.iloc[-1], d.iloc[-1]
print(f"{'Model':<28}{'Misalign':>10}{'Fair INR':>10}")
print(f"{'REER structural':<28}{r2.misalignment_pct:>+9.1f}%{r2.inr_fair_reer:>10.2f}")
print(f"{'FEER static':<28}{r3.feer_static_pct:>+9.1f}%{r3.inr_fair_static:>10.2f}")
print(f"{'FEER conditional':<28}{r3.feer_cond_pct:>+9.1f}%{r3.inr_fair_cond:>10.2f}")
print(f"{'BEER short-run':<28}{r4.beer_misalign:>+9.1f}%{r4.inr_fair_beer:>10.2f}")
print(f"\nFinancing premium: {r3.financing_premium:+.1f}pp")
print(f"P(stress): {r4.p_stress:.2f}")
